# 03 — Integração e limpeza

**Objetivo:** juntar em uma única base, por município de SP:
- idosos sozinhos / domicílios com responsável idoso (notebook 01)
- internações por causa e ano (notebook 02)
- IDH municipal (controle)

**Como as bases são cruzadas:** por **nome de município normalizado**
(`config.normalizar_municipio`), não por código IBGE/DATASUS — nenhuma das
fontes reais que conseguimos traz os dois códigos ao mesmo tempo, e não
temos acesso à internet neste ambiente para baixar a lista oficial de
códigos do IBGE. O cruzamento por nome foi validado (ver notebook 02).

**Duas saídas desta etapa, com propósitos diferentes:**
1. `dataset_consolidado_sp.csv` — **painel** (uma linha por município × ano,
   3.225 linhas). Bom para gráficos de evolução e para o estudo de caso de
   Rio Claro (notebook 05).
2. `dataset_municipios_sp.csv` — **nível município** (uma linha por
   município, 645 linhas, internações somadas 2022-2026). **É esta que usamos
   para testar a hipótese principal no notebook 04** — ver a nota abaixo.

⚠️ **Por que duas tabelas, e não só o painel:** `pct_idosos_sozinhos` vem do
Censo 2022 — é o **mesmo valor nos 5 anos** de cada município. Se você rodar
a correlação/regressão direto no painel (3.225 linhas), cada município entra
5 vezes com a mesma variável independente — isso é
[pseudorreplicação](https://en.wikipedia.org/wiki/Pseudoreplication): infla
artificialmente o "n" e faz resultados parecerem mais significativos do que
são. A tabela por município (645 linhas, uma observação independente por
município) é a forma estatisticamente correta de testar a hipótese.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd


## 3.1 Carregar as bases já processadas


In [ ]:
censo = pd.read_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv")
internacoes = pd.read_csv(config.DATA_PROCESSED / "internacoes_sp.csv")

print("censo:", censo.shape, "-> municípios:", censo["municipio_norm"].nunique())
print("internacoes:", internacoes.shape, "-> municípios:", internacoes["municipio_norm"].nunique())


## 3.2 IDH municipal (controle)

`data/external/idh_sp.csv` (colunas `municipio`, `idh`) — ver
`data/external/FONTES_RIO_CLARO.md` para a origem exata dos valores.

⚠️ **Cobertura parcial: 259 dos 645 municípios de SP têm IDH nesse arquivo**
(uma 260ª linha, "Guaxupé", nem é de SP — é de MG, provavelmente um erro no
arquivo de origem, e cai fora do cruzamento sem problema). Os municípios
sem IDH entram como `NaN` — a regressão do notebook 04 (`statsmodels`)
descarta essas linhas automaticamente.

Essa variável entra como **controle**: o objetivo é checar se a associação
entre "idosos sozinhos" e "internações" se mantém mesmo depois de levar em
conta a renda/desenvolvimento do município.


In [ ]:
caminho_idh = config.DATA_EXTERNAL / "idh_sp.csv"

idh = pd.read_csv(caminho_idh).rename(columns={"idh": "idhm"})
idh["municipio_norm"] = idh["municipio"].apply(config.normalizar_municipio)
idh = idh[["municipio_norm", "idhm"]]

print(f"{len(idh)} municípios com IDH no arquivo (de 645 no estado)")
idh.head()


## 3.3 Montar o painel (município × ano)

O merge parte da lista **completa** de municípios (os 645 do Censo,
cruzados com todos os anos do SIH) e preenche com 0 onde não há registro de
internação — em vez de partir de `internacoes`, o que faria os municípios
sem nenhuma internação registrada desaparecerem da base em vez de entrar
com 0.


In [ ]:
internacoes_wide = (
    internacoes
    .pivot_table(index=["municipio_norm", "ano"], columns="causa", values="internacoes", fill_value=0)
    .reset_index()
)
causas_cols = list(config.CAUSAS_SIH.keys())

anos_sih = sorted(internacoes["ano"].unique())
grid = censo[["municipio", "municipio_norm"]].merge(pd.DataFrame({"ano": anos_sih}), how="cross")

painel = grid.merge(internacoes_wide, on=["municipio_norm", "ano"], how="left")
painel[causas_cols] = painel[causas_cols].fillna(0)
painel["total_internacoes_causas_estudo"] = painel[causas_cols].sum(axis=1)

painel = painel.merge(censo, on=["municipio", "municipio_norm"], how="left")
painel = painel.merge(idh, on="municipio_norm", how="left")

# Taxa por 100 mil DOMICÍLIOS com responsável idoso (não por 100 mil idosos-pessoa --
# ver a nota metodológica do notebook 01 sobre a diferença)
painel["taxa_internacao_100k_domicilios_idosos"] = (
    painel["total_internacoes_causas_estudo"] / painel["domicilios_resp_idoso"] * 100_000
)

print(painel.shape, "(esperado 645 x 5 = 3225)")
print("linhas com IDH:", painel["idhm"].notna().sum())
painel.head()


In [ ]:
painel.to_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "dataset_consolidado_sp.csv")


## 3.4 Montar a base por município (nível de análise da hipótese principal)

Uma linha por município, com as internações **somadas** entre 2022 e 2026
(nota: 2026 é parcial, até julho) e as variáveis que não mudam por ano
(`pct_idosos_sozinhos`, `idhm`) entrando uma única vez.


In [ ]:
municipios_agg = painel.groupby(["municipio", "municipio_norm"]).agg(
    total_internacoes=("total_internacoes_causas_estudo", "sum"),
    domicilios_resp_idoso=("domicilios_resp_idoso", "first"),
    idosos_sozinhos=("idosos_sozinhos", "first"),
    pct_idosos_sozinhos=("pct_idosos_sozinhos", "first"),
    idhm=("idhm", "first"),
).reset_index()

municipios_agg["taxa_internacao_100k_domicilios_idosos"] = (
    municipios_agg["total_internacoes"] / municipios_agg["domicilios_resp_idoso"] * 100_000
)

print(municipios_agg.shape, "(esperado 645)")
municipios_agg.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "dataset_municipios_sp.csv")


## 3.5 Destacar Rio Claro


In [ ]:
print("--- Painel (por ano) ---")
display(painel[painel["municipio"] == config.RIO_CLARO_NOME])

print("--- Agregado (nível município) ---")
display(municipios_agg[municipios_agg["municipio"] == config.RIO_CLARO_NOME])
